# Predicting extubation failure

Logistic regression (interpretable model) and XGBoost (black-box model) predict recorded reintubation or death within 48 hours of planned extubation. Comparators are a prevalence-only model, a random score and an exploratory rapid shallow breathing index (RSBI-summary ≥ 105) rule. Inputs are the patient-level CSVs from the phenotyping notebook. The notebook validates inputs, creates temporal partitions, and fits and evaluates models. Imputation and encoding are learned within training folds.

| Notebook section | Report section |
|---|---|
| 2–3 Settings, inputs, temporal split | Methods: study design and data split |
| 4 Cohort description | Methods/Results: timeline figure, Table 1, six-hour mean distributions |
| 5–6 Models, tuning, calibration, metrics | Methods: machine learning and evaluation |
| 7 Development results and sensitivity analyses | Results: internal validation, sensitivity analyses |
| 8 Interpretation | Results/Discussion: feature importance |
| 9 Final evaluation | Results: temporal validation |


## 1. Setup

Use Python 3.12. These package versions were tested together. If needed, install once in a notebook cell, then restart the kernel:

```python
%pip install numpy==2.4.4 pandas==3.0.2 scipy==1.17.1 scikit-learn==1.8.0 xgboost==3.2.0 matplotlib==3.10.8 joblib==1.5.3
```


In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import hashlib
import importlib.metadata
import json
import platform
import re
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
    confusion_matrix, log_loss, precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import DMatrix, XGBClassifier

from IPython.display import display, Image

# Locate the project folder (the one containing data/) from wherever the notebook is opened.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook within the group project folder containing data/.")


## 2. Settings

Edit `CONFIG`, then run all cells from top to bottom.

| Analysis | Cohort | Predictors |
|---|---|---|
| Main: `filtered_six_hour` | Supplied filtered cohort | 19 six-hour means and five static covariates |
| Missingness sensitivity: `full_six_hour` | Full eligible cohort | Same predictors |
| Feature sensitivity: `filtered_last_hour` | Same patients and folds as main | Last available hourly summaries and the same static covariates |

1. `mode="trial"`: reduced grids and 2 × 2 CV on real development patients to test the flow. The test set stays closed.
2. `mode="main"`: full grids and 5 outer / 3 inner folds. Enable `run_final_test=True` only after all analysis choices are fixed.

Fewer than 20 test events or non-events triggers a warning, not an exclusion or skipped evaluation. Small samples give unstable estimates; metrics requiring both classes are undefined when one is absent. The warning threshold is a reporting choice, not a formal adequacy criterion.

Outputs: `models/outputs/<mode>/<analysis>/`; combined tables and figures: `models/outputs/<mode>/`.


In [ ]:
CONFIG = {
    "execution": {
        "mode": "trial",            # trial (reduced) or main
        "run_final_test": False,    # main mode only; enable after choices are frozen
    },
    "data_files": {
        "filtered": str(PROJECT_ROOT / "data/average_6h_filtered_dataset.csv"),
        "full": str(PROJECT_ROOT / "data/average_6h_full_dataset.csv"),
    },
    "coverage_csv": str(PROJECT_ROOT / "data/average_6h_coverage.csv"),
    "last_hour_csv": str(PROJECT_ROOT / "data/clean_full_final.csv"),
    "metadata_csv": None,  # optional sidecar if temporal fields are absent from the CSVs
    "output_root": str(PROJECT_ROOT / "models/outputs"),
    "include_last_hour_ablation": True,
    "target": "combined_outcome",
    "seed": 42,
    "split_mode": "temporal",
    "temporal_cutoff": 2020,  # same calendar-year boundary for every analysis
    "min_test_events": 20,    # warning only; all test patients are evaluated
    "feature_columns": None,  # None: use recognised physiological predictors; audit is saved
    "categorical_columns": ["gender", "sex", "first_careunit", "icu_type_at_admission"],
    "rsbi_column": None,      # None: use the RSBI summary available in each feature set
    "negative_followup_policy": "retain_recorded_labels",  # report uncertainty; do not redefine the cohort

    # Nested cross-validation and exhaustive grids (GridSearchCV).
    "outer_folds": 5,
    "inner_folds": 3,
    "lr_c_values": [0.001, 0.01, 0.1, 1, 10, 100],
    "lr_max_iter": 3000,
    "xgb_param_grid": {       # 2 x 3 x 2 x 2 x 2 = 48 combinations
        "n_estimators": [200, 400],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.1],
        "min_child_weight": [1, 5],
        "reg_lambda": [1, 10],
    },
    "xgb_subsample": 0.8,
    "xgb_colsample_bytree": 0.8,
    "xgb_threads": 4,

    # Evaluation.
    "calibrate": True,
    "calibration_method": "sigmoid",
    "sensitivity_target": 0.80,            # exploratory threshold target, not clinically validated
    "decision_curve_max_threshold": 0.30,  # decision curve covers risk thresholds 1% to this value
    "bootstrap_replicates": 1000,

    # Trial mode only changes the tuning budget, never the patients or split.
    "trial_overrides": {
        "outer_folds": 2,
        "inner_folds": 2,
        "lr_c_values": [0.1, 1],
        "xgb_param_grid": {
            "n_estimators": [30],
            "max_depth": [2, 3],
            "learning_rate": [0.1],
            "min_child_weight": [1],
            "reg_lambda": [1],
        },
        "xgb_threads": 1,
        "bootstrap_replicates": 50,
    },
}


## 3. Inputs and temporal split

Files in `data/`:
- `average_6h_filtered_dataset.csv`: main cohort, before median imputation.
- `average_6h_full_dataset.csv`: full-cohort sensitivity.
- `average_6h_coverage.csv`: observation counts and supplied filter membership; validation only, never predictors.
- `clean_full_final.csv`: bin 5 for the optional feature sensitivity, joined to exactly the main-cohort patients. Static covariates, index events and labels must agree.

Each patient needs `extubation_time`, `anchor_year`, and `anchor_year_group`. Include these in both averaged CSVs, or set `metadata_csv` to a file with one row per `subject_id` and these fields (preferably `stay_id` too). **The current exports omit these fields; temporal modelling stops until they are supplied.**

Predictors cover `[t−6 h, t)`, prediction is at planned extubation `t`, and the outcome window is `(t, t+48 h]`, with no gap. Eligibility and exclusions, including unplanned extubation, come from the phenotyping notebook. Supplied patients and labels are retained; imputation is learned inside training folds.

MIMIC-IV dates are shifted per patient. Map the extubation year to its possible calendar-year interval using `anchor_year` and `anchor_year_group`. Entirely pre-2020 intervals form development; intervals entirely from 2020 form test. Intervals crossing the cutoff are excluded because their temporal order is uncertain. The same cutoff and partition apply to every analysis.

The notebook checks duplicate IDs/columns, labels, shared rows, coverage and broad physiological ranges. It reports implausible values without changing them. `hospital_observed_48h` is follow-up metadata, never a predictor; incomplete observation among negatives is reported.


In [ ]:
REQUIRED_CONFIG = {
    "data_csv", "metadata_csv", "output_dir", "target", "seed", "split_mode",
    "temporal_cutoff", "min_test_events", "feature_columns", "categorical_columns",
    "rsbi_column", "outer_folds", "inner_folds", "lr_c_values", "lr_max_iter",
    "xgb_param_grid", "xgb_subsample", "xgb_colsample_bytree", "xgb_threads",
    "calibrate", "calibration_method", "sensitivity_target", "decision_curve_max_threshold",
    "bootstrap_replicates", "run_mode", "analysis", "feature_set", "coverage_csv", "last_hour_csv",
    "suite_output_dir", "negative_followup_policy", "pipeline_version",
}

PHYSIOLOGY = {
    "heart_rate", "mbp", "sbp", "dbp", "resp_rate", "temperature", "spo2",
    "respiratory_rate_total", "respiratory_rate_spontaneous",
    "tidal_volume_observed", "tidal_volume_spontaneous", "fio2", "peep",
    "plateau_pressure", "minute_volume", "minute_ventilation", "po2", "pco2",
    "ph", "fio2_bg", "fio2_chartevents", "pao2fio2ratio", "bicarbonate",
    "baseexcess", "lactate", "rsbi",
}
BASELINE = {"age", "admission_age", "gender", "sex", "first_careunit", "icu_type_at_admission",
    "icu_hours_before_extubation", "hospital_hours_before_extubation"}
OUTCOME_COMPONENTS = ["death_within_48_hours", "reintubation_within_48_hours"]
MODEL_NAMES = ["logistic_regression", "xgboost"]
PROBABILITY_MODELS = MODEL_NAMES + ["prevalence_baseline"]

# Broad limits for error detection only: values outside them are counted and reported, never changed.
# Units follow the extraction notebook: FiO2 as a fraction, tidal volume in mL, temperature in degrees C.
PLAUSIBLE_RANGES = {
    "heart_rate": (20, 250), "mbp": (20, 200), "sbp": (40, 300), "dbp": (10, 200),
    "resp_rate": (1, 80), "respiratory_rate_total": (1, 80), "respiratory_rate_spontaneous": (1, 80),
    "temperature": (30, 43), "spo2": (50, 100),
    "tidal_volume_observed": (50, 2000), "tidal_volume_spontaneous": (50, 2000),
    "fio2": (0.21, 1.0), "fio2_bg": (0.21, 1.0), "fio2_chartevents": (0.21, 1.0),
    "peep": (0, 30), "plateau_pressure": (0, 60), "minute_volume": (0.5, 40), "minute_ventilation": (0.5, 40),
    "po2": (20, 700), "pco2": (10, 150), "ph": (6.8, 7.8), "pao2fio2ratio": (20, 1000),
    "bicarbonate": (2, 60), "baseexcess": (-40, 40), "lactate": (0.1, 30), "rsbi": (1, 1000),
    "age": (18, 110), "admission_age": (18, 110),
}


def validate_config(config):
    """Reject missing, unknown or inconsistent settings before any data are read."""
    cfg = deepcopy(config)
    missing = sorted(REQUIRED_CONFIG - set(cfg))
    if missing:
        raise ValueError("CONFIG is missing: " + str(missing))
    unknown = sorted(set(cfg) - REQUIRED_CONFIG)
    if unknown:
        raise ValueError("Unknown CONFIG settings: " + str(unknown))
    if cfg["run_mode"] not in {"main", "trial"}:
        raise ValueError("CONFIG['execution']['mode'] must be 'trial' or 'main'.")
    if cfg["calibration_method"] not in {"sigmoid", "isotonic"}:
        raise ValueError("calibration_method must be 'sigmoid' or 'isotonic'.")
    if cfg["negative_followup_policy"] not in {"retain_recorded_labels", "require_complete_negatives"}:
        raise ValueError("Unknown negative_followup_policy.")
    if cfg["split_mode"] != "temporal" or cfg["temporal_cutoff"] is None:
        raise ValueError("These analyses require a common, explicit temporal cutoff.")
    if int(cfg["min_test_events"]) < 1:
        raise ValueError("min_test_events must be at least 1.")
    if not 0 < cfg["decision_curve_max_threshold"] < 1:
        raise ValueError("decision_curve_max_threshold must be between 0 and 1.")
    if int(cfg["bootstrap_replicates"]) < 10:
        raise ValueError("bootstrap_replicates must be at least 10; intervals are always reported.")
    return cfg


def make_run_configs(config):
    """Give every cohort/feature analysis its own folder within the selected mode."""
    cfg = deepcopy(config)
    execution = cfg.pop("execution", None)
    trial_overrides = cfg.pop("trial_overrides", None)
    files = cfg.pop("data_files")
    root = Path(cfg.pop("output_root"))
    include_ablation = cfg.pop("include_last_hour_ablation")
    if not isinstance(execution, dict) or set(execution) != {"mode", "run_final_test"}:
        raise ValueError("CONFIG['execution'] must define mode and run_final_test.")
    mode = execution["mode"]
    if mode not in {"main", "trial"}:
        raise ValueError("CONFIG['execution']['mode'] must be 'trial' or 'main'.")
    if execution["run_final_test"] and mode == "trial":
        raise ValueError("Trial runs cannot evaluate the real holdout. Use main after freezing all analyses.")
    if mode == "trial":
        overrides = deepcopy(trial_overrides)
        allowed = {"outer_folds", "inner_folds", "xgb_threads", "lr_c_values",
            "xgb_param_grid", "bootstrap_replicates"}
        if not overrides or set(overrides) - allowed:
            raise ValueError("Trial overrides must specify reduced tuning settings, not change patients or the split.")
        cfg.update(overrides)
    cfg["run_mode"] = mode
    cfg["pipeline_version"] = "six-hour-means-3"
    cfg["suite_output_dir"] = str(root / mode)
    specs = [("filtered_six_hour", "filtered", "six_hour"),
             ("full_six_hour", "full", "six_hour")]
    if include_ablation:
        specs.append(("filtered_last_hour", "filtered", "last_hour"))
    return {name: validate_config({**deepcopy(cfg), "analysis": name,
        "feature_set": feature_set, "data_csv": files[cohort],
        "output_dir": str(root / mode / name)}) for name, cohort, feature_set in specs}


def write_json(path, value):
    """Save settings, converting NumPy scalars and paths to JSON values."""
    def encode(x):
        if isinstance(x, np.generic):
            return x.item()
        if isinstance(x, Path):
            return str(x)
        raise TypeError(f"Cannot encode {type(x)}")
    Path(path).write_text(json.dumps(value, indent=2, default=encode), encoding="utf-8")


def file_hash(path):
    """SHA-256 of a file, used to detect changed inputs or models."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def binary_series(s, name):
    """Require observed binary labels."""
    out = pd.to_numeric(s, errors="raise")
    if out.isna().any() or not out.isin([0, 1]).all():
        raise ValueError(f"{name} must contain only observed 0/1 labels.")
    return out.astype(int)


def forbidden_feature(name, target):
    """Identifiers, outcomes and post-prediction times must never be predictors."""
    n = name.lower()
    return (n == target.lower() or n in {"subject_id", "hadm_id", "stay_id", "dod", "label", "y"}
            or bool(re.search(r"outcome|death|deathtime|reintubat|discharg|dischtime|outtime|expire|los_|length_of_stay|followup|follow_up|hospital_observed|anchor_year|extubation_time|admittime|icu_intime|charttime|storetime|specimen_id|observed_values|missingness|retained_by", n)))


def feature_family(name):
    """Map an hourly bin or window summary (e.g. RSBI_bin_5) to its variable (rsbi)."""
    return re.sub(r"(_bin_[0-5]|_(mean|std|sd|min|max|last|slope|count|missing)(?:_6h)?|_6h)$", "", name.lower())


def recognised_feature(name):
    """Static covariates and physiological variables are the default predictors."""
    return name.lower() in BASELINE or feature_family(name) in PHYSIOLOGY


def range_audit(df, columns):
    """Count predictor values outside PLAUSIBLE_RANGES; the data are not modified."""
    rows = []
    for c in columns:
        limits = PLAUSIBLE_RANGES.get(feature_family(c))
        if limits is None or not pd.api.types.is_numeric_dtype(df[c]):
            continue
        v = df[c].dropna()
        rows.append({"column": c, "lower_limit": limits[0], "upper_limit": limits[1],
            "observed": len(v), "below": int((v < limits[0]).sum()), "above": int((v > limits[1]).sum()),
            "min": v.min() if len(v) else np.nan, "max": v.max() if len(v) else np.nan})
    audit = pd.DataFrame(rows, columns=["column", "lower_limit", "upper_limit", "observed", "below", "above", "min", "max"])
    audit["implausible_fraction"] = (audit.below + audit.above) / audit.observed.replace(0, np.nan)
    return audit


def race_group(s):
    """Collapse MIMIC race strings into broad groups for description and subgroup checks only."""
    s = s.fillna("UNKNOWN").astype(str).str.upper()
    return np.select([s.str.contains("WHITE"), s.str.contains("BLACK"),
        s.str.contains("HISPANIC|LATINO"), s.str.contains("ASIAN")],
        ["White", "Black", "Hispanic/Latino", "Asian"], default="Other/unknown")


In [ ]:
def load_cohort(cfg):
    """Read one cohort CSV, check it and select predictors. Predictor values are not altered."""
    path = Path(cfg["data_csv"])
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path.resolve()}. Set data_csv to the full cohort before median imputation and splitting.")
    if path.stem.startswith(("train_set", "test_set", "X_train", "X_test")):
        raise ValueError("Use the full cohort before median imputation and splitting.")
    # Check duplicate headers before pandas renames them.
    with path.open(newline="", encoding="utf-8-sig") as f:
        header = next(csv.reader(f))
    if len(header) != len(set(header)):
        raise ValueError("Duplicate CSV column names; fix the export.")
    df = pd.read_csv(path, dtype={"subject_id": "string", "stay_id": "string", "hadm_id": "string"})
    if "subject_id" not in df or cfg["target"] not in df:
        raise ValueError("Required columns: subject_id and " + cfg["target"])
    if df.subject_id.isna().any() or df.subject_id.duplicated().any():
        raise ValueError("Expected ONE row per subject_id, with a single index extubation.")
    # Keep splits reproducible if input rows are reordered.
    df = df.sort_values("subject_id").reset_index(drop=True)
    df[cfg["target"]] = binary_series(df[cfg["target"]], cfg["target"])
    components_present = [c in df for c in OUTCOME_COMPONENTS]
    if all(components_present):
        for c in OUTCOME_COMPONENTS:
            df[c] = binary_series(df[c], c)
        if not np.array_equal(df[cfg["target"]], df[OUTCOME_COMPONENTS].max(axis=1)):
            raise ValueError("combined_outcome disagrees with death OR reintubation.")
    else:
        warnings.warn("Outcome components are missing; cannot verify the combined label.")
    last_hour_path = None
    if cfg["feature_set"] == "last_hour":
        # Keep the averaged main cohort; replace only its physiological summaries.
        last_hour_path = Path(cfg["last_hour_csv"])
        hourly = pd.read_csv(last_hour_path, dtype={"subject_id": "string", "stay_id": "string", "hadm_id": "string"})
        if hourly.subject_id.isna().any() or hourly.subject_id.duplicated().any():
            raise ValueError("The hourly CSV must have one non-missing row per subject_id.")
        if not set(df.subject_id).issubset(set(hourly.subject_id)):
            raise ValueError("The hourly CSV does not cover every main-cohort patient.")
        hourly = hourly.set_index("subject_id").loc[df.subject_id].reset_index()
        matched = [c for c in ["stay_id", "hadm_id", cfg["target"], *OUTCOME_COMPONENTS,
            "hospital_observed_48h", *sorted(BASELINE)] if c in df]
        if not set(matched).issubset(hourly.columns):
            raise ValueError("The hourly CSV lacks the fields needed to verify the same patients/index events.")
        try:
            pd.testing.assert_frame_equal(df[matched], hourly[matched], check_dtype=False)
        except AssertionError:
            raise ValueError("Hourly and averaged CSVs disagree in index events, static covariates or outcomes.") from None
        means = [c for c in df if c.endswith("_mean_6h") and feature_family(c) in PHYSIOLOGY]
        bins = [c for c in hourly if c.endswith("_bin_5") and feature_family(c) in PHYSIOLOGY]
        if {feature_family(c) for c in means} != {feature_family(c) for c in bins}:
            raise ValueError("Six-hour and last-hour inputs must cover the same physiological variables.")
        df = pd.concat([df.drop(columns=means), hourly[bins]], axis=1)
    metadata_path = Path(cfg["metadata_csv"]) if cfg.get("metadata_csv") else None
    if metadata_path:
        if not metadata_path.exists():
            raise FileNotFoundError(f"Metadata CSV not found: {metadata_path}")
        meta = pd.read_csv(metadata_path, dtype={"subject_id": "string", "stay_id": "string", "hadm_id": "string"})
        if "subject_id" not in meta or meta.subject_id.isna().any() or meta.subject_id.duplicated().any():
            raise ValueError("Metadata must have one non-missing row per subject_id.")
        # Overlapping columns must agree before merging metadata.
        both = sorted((set(meta.columns) & set(df.columns)) - {"subject_id"})
        joined = df[["subject_id"] + both].merge(meta, on="subject_id", how="left", validate="one_to_one", suffixes=("_main", "_meta"))
        for c in both:
            a, b = joined[c + "_main"], joined[c + "_meta"]
            if not a.astype("string").fillna("<NA>").eq(b.astype("string").fillna("<NA>")).all():
                raise ValueError(f"Main CSV and metadata disagree in {c}; resolve before modelling.")
        extra = [c for c in meta if c not in df]
        df = df.merge(meta[["subject_id"] + extra], on="subject_id", how="left", validate="one_to_one")
    followup_col = next((c for c in ["hospital_observed_48h", "followup_complete"] if c in df), None)
    if followup_col:
        original = df[followup_col]
        known = pd.to_numeric(original.replace({True: 1, False: 0, "True": 1, "False": 0,
            "true": 1, "false": 0}), errors="coerce")
        if (original.notna() & (known.isna() | ~known.isin([0, 1]))).any():
            raise ValueError(f"{followup_col} must contain 0/1, booleans or missing values.")
        df[followup_col] = known
    else:
        known = pd.Series(np.nan, index=df.index)
    incomplete_negative = df[cfg["target"]].eq(0) & known.ne(1)
    if cfg["negative_followup_policy"] == "require_complete_negatives" and incomplete_negative.any():
        raise ValueError("Negative labels lack complete follow-up. Resolve the phenotype; no rows were silently dropped.")
    requested = cfg.get("feature_columns")
    selected = list(requested) if requested is not None else [c for c in df if recognised_feature(c)]
    if not selected or len(set(selected)) != len(selected):
        raise ValueError("No recognised predictors or duplicate features. Set feature_columns explicitly.")
    if not set(selected).issubset(df.columns):
        raise ValueError("Feature columns missing: " + str(sorted(set(selected) - set(df.columns))))
    leaks = [c for c in selected if forbidden_feature(c, cfg["target"])]
    if leaks:
        raise ValueError("Forbidden identifier/outcome/future predictors: " + str(leaks))
    cats = [c for c in selected if c in cfg["categorical_columns"]]
    nums = [c for c in selected if c not in cats]
    if not nums:
        raise ValueError("No numeric physiological predictors selected.")
    # Type conversion only: infinite values cannot be imputed, so they are treated as missing.
    for c in nums:
        df[c] = pd.to_numeric(df[c], errors="raise").replace([np.inf, -np.inf], np.nan)
    for c in cats:
        df[c] = df[c].astype(object).where(df[c].notna(), np.nan)
    rsbi_candidates = [cfg["rsbi_column"]] if cfg.get("rsbi_column") else ["RSBI_bin_5", "rsbi_bin_5", "RSBI_mean_6h", "rsbi_mean_6h", "RSBI", "rsbi"]
    rsbi_col = next((c for c in rsbi_candidates if c in df), None)
    if cfg.get("rsbi_column") and rsbi_col is None:
        raise ValueError("Configured rsbi_column is absent.")
    if rsbi_col:
        df[rsbi_col] = pd.to_numeric(df[rsbi_col], errors="raise").replace([np.inf, -np.inf], np.nan)
    else:
        warnings.warn("No RSBI column; skipping the RSBI comparison.")
    audit = pd.DataFrame({"column": df.columns})
    audit["used_as_predictor"] = audit.column.isin(selected)
    audit["reason"] = ["predictor" if c in selected else "blocked ID/outcome/time/follow-up" if forbidden_feature(c, cfg["target"]) else "metadata or unrecognised: not selected" for c in audit.column]
    hashes = {"data_csv_sha256": file_hash(path)}
    if last_hour_path:
        hashes["last_hour_csv_sha256"] = file_hash(last_hour_path)
    hashes["coverage_csv_sha256"] = file_hash(cfg["coverage_csv"])
    if metadata_path and metadata_path.exists():
        hashes["metadata_csv_sha256"] = file_hash(metadata_path)
    return df, nums, cats, rsbi_col, audit, hashes


def followup_summary(df, cfg):
    """Count labels without complete 48-hour hospital observation."""
    col = next((c for c in ["hospital_observed_48h", "followup_complete"] if c in df), None)
    observed = df[col].eq(1) if col else pd.Series(False, index=df.index)
    return {"followup_field": col, "patients": len(df),
        "negative_labels_without_complete_observation": int((df[cfg["target"]].eq(0) & ~observed).sum()),
        "known_events_without_complete_observation": int((df[cfg["target"]].eq(1) & ~observed).sum()),
        "policy": cfg["negative_followup_policy"]}


In [ ]:
def make_split(df, cfg):
    """Assign each patient to development, test or ambiguous_time (excluded) by real calendar year."""
    manifest = df[["subject_id"]].copy()
    manifest["partition"] = "ambiguous_time"
    if cfg["split_mode"] == "temporal":
        required = ["extubation_time", "anchor_year", "anchor_year_group"]
        missing = [c for c in required if c not in df]
        if missing:
            raise ValueError("Temporal holdout needs " + str(missing) + ". Include them in data_csv or metadata_csv, keyed by subject_id.")
        groups = df.anchor_year_group.astype(str).str.extract(r"^\s*(\d{4})\s*[-–]\s*(\d{4})\s*$").astype(float)
        anchor = pd.to_numeric(df.anchor_year, errors="coerce")
        year = pd.to_datetime(df.extubation_time, errors="coerce").dt.year
        if groups.isna().any().any() or anchor.isna().any() or year.isna().any():
            raise ValueError("Temporal metadata contain missing or invalid values.")
        if (groups[1] < groups[0]).any() or (anchor % 1 != 0).any():
            raise ValueError("Invalid anchor year or year-group interval.")
        offset = year - anchor
        # Map shifted extubation years to possible calendar-year intervals.
        manifest["possible_year_lower"] = (groups[0] + offset).astype(int)
        manifest["possible_year_upper"] = (groups[1] + offset).astype(int)
        cutoff = cfg["temporal_cutoff"]
        manifest.loc[manifest.possible_year_upper < cutoff, "partition"] = "development"
        manifest.loc[manifest.possible_year_lower >= cutoff, "partition"] = "test"
        split_note = {"mode": "temporal", "cutoff": int(cutoff), "rule": "development: upper possible year < cutoff; test: lower possible year >= cutoff; overlapping intervals excluded", "cutoff_choice": "explicit, chosen without model performance"}
    else:
        raise ValueError("split_mode must be temporal.")
    dev = np.flatnonzero(manifest.partition.eq("development"))
    test = np.flatnonzero(manifest.partition.eq("test"))
    if not len(dev) or not len(test):
        raise ValueError("Empty development/test set. Check year intervals and temporal_cutoff.")
    assert not set(df.iloc[dev].subject_id) & set(df.iloc[test].subject_id)
    # Validate fold sizes using development labels only.
    checked_cv(df.iloc[dev][cfg["target"]].to_numpy(), cfg["outer_folds"], cfg["seed"])
    return manifest, dev, test, split_note


def checked_cv(y, folds, seed):
    """Stratified folds, after checking each class has enough patients."""
    counts = np.bincount(np.asarray(y, dtype=int), minlength=2)
    if folds < 2 or min(counts) < folds:
        raise ValueError(f"Need at least {folds} patients in each class for cross-validation; found {counts.tolist()}.")
    return list(StratifiedKFold(folds, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))


def holdout_status(df, test_idx, cfg):
    """Count test patients and events only; no model is applied to the holdout here."""
    y = df.iloc[test_idx][cfg["target"]]
    events, non_events = int(y.sum()), int(len(y) - y.sum())
    return {"test_patients": len(y), "test_events": events,
        "test_non_events": non_events,
        "small_sample_warning": min(events, non_events) < cfg["min_test_events"],
        "rule": f"warn below {cfg['min_test_events']} events or non-events; retain all test patients"}


def warn_small_holdout(status, cfg):
    """Low event counts affect precision, not eligibility for evaluation."""
    if status["small_sample_warning"]:
        warnings.warn(f"{cfg['analysis']}: temporal holdout has {status['test_events']} events and "
            f"{status['test_non_events']} non-events (warning threshold: {cfg['min_test_events']}). "
            "All test patients are retained. Estimates and bootstrap intervals may be unstable; "
            "metrics requiring both outcome classes are undefined if one class is absent.")


def check_inputs(config):
    """Check inputs, fold sizes and holdout size without training or scoring test patients."""
    cfg = validate_config(config)
    df, nums, cats, rsbi_col, _, _ = load_cohort(cfg)
    manifest, dev, test, _ = make_split(df, cfg)
    y = df.iloc[dev][cfg["target"]].to_numpy()
    for fold, (train, _) in enumerate(checked_cv(y, cfg["outer_folds"], cfg["seed"])):
        checked_cv(y[train], cfg["inner_folds"], cfg["seed"] + fold + 1)
    holdout = holdout_status(df, test, cfg)
    warn_small_holdout(holdout, cfg)
    audit = range_audit(df, nums)
    return {"mode": cfg["run_mode"], "patients": len(df), "development_patients": len(dev),
        "development_events": int(y.sum()), "test_patients": holdout["test_patients"],
        "test_events": holdout["test_events"], "test_non_events": holdout["test_non_events"],
        "small_holdout_warning": holdout["small_sample_warning"],
        "excluded_ambiguous_year": int(manifest.partition.eq("ambiguous_time").sum()),
        "predictors": len(nums) + len(cats), "events_per_predictor": round(y.sum() / (len(nums) + len(cats)), 2),
        "values_outside_plausible_range": int(audit.below.sum() + audit.above.sum()),
        "rsbi_comparator": rsbi_col, "outer_folds": cfg["outer_folds"], "inner_folds": cfg["inner_folds"],
        "calibration": cfg["calibration_method"] if cfg["calibrate"] else "disabled",
        "development_negatives_without_complete_followup": followup_summary(df.iloc[dev], cfg)["negative_labels_without_complete_observation"]}


def validate_coverage(filtered, full, coverage_csv):
    """Check the supplied missingness audit; do not apply a new patient filter."""
    coverage = pd.read_csv(coverage_csv, dtype={"subject_id": "string", "stay_id": "string"})
    if coverage.subject_id.isna().any() or coverage.subject_id.duplicated().any():
        raise ValueError("Coverage must contain one row per subject_id.")
    if set(coverage.subject_id) != set(full.subject_id):
        raise ValueError("Coverage and full cohort must contain the same patients.")
    coverage = coverage.set_index("subject_id").loc[full.subject_id].reset_index()
    if not coverage.stay_id.equals(full.stay_id):
        raise ValueError("Coverage and full cohort refer to different ICU stays.")
    retained = coverage.retained_by_missingness_filter.map({True: True, False: False,
        "True": True, "False": False, "true": True, "false": False})
    if retained.isna().any() or set(coverage.loc[retained, "subject_id"]) != set(filtered.subject_id):
        raise ValueError("Coverage retention flags do not match the supplied filtered cohort.")
    means = [c for c in full if c.endswith("_mean_6h") and feature_family(c) in PHYSIOLOGY]
    if not means:
        raise ValueError("No six-hour mean features found in the averaged cohort CSVs.")
    if not np.allclose(full[means].isna().mean(axis=1), coverage.average_feature_missingness):
        raise ValueError("Coverage missingness disagrees with the averaged features.")
    for c in means:
        count = c.removesuffix("_mean_6h") + "_observed_values_6h"
        if count not in coverage:
            raise ValueError(f"Coverage is missing {count}.")
        observed = pd.to_numeric(coverage[count], errors="raise")
        if observed.isna().any() or (observed < 0).any() or (observed % 1 != 0).any():
            raise ValueError(f"Coverage counts must be non-negative integers: {count}.")
        if not np.array_equal(observed.eq(0), full[c].isna()):
            raise ValueError(f"Coverage counts disagree with missing values in {c}.")
    return {"full_patients": len(full), "filtered_patients": len(filtered),
        "removed_by_supplied_filter": len(full) - len(filtered), "mean_features": len(means)}


def prepare_analyses(configs):
    """Lock shared predictors and check that cohorts represent the same index events."""
    configs = deepcopy(configs)
    main = configs["filtered_six_hour"]
    filtered, nums, cats, _, _, _ = load_cohort(main)
    features = nums + cats
    full, full_nums, full_cats, _, _, _ = load_cohort(configs["full_six_hour"])
    coverage_summary = validate_coverage(filtered, full, main["coverage_csv"])
    print("Six-hour input validation:", coverage_summary)
    if set(features) != set(full_nums + full_cats):
        raise ValueError("Filtered and full cohorts must have the same predictor definitions.")
    if not set(filtered.subject_id).issubset(set(full.subject_id)):
        raise ValueError("The filtered cohort must be a subset of the full eligible cohort.")
    shared = full.set_index("subject_id").loc[filtered.subject_id].reset_index()
    optional = ["stay_id", "hadm_id", *OUTCOME_COMPONENTS, "hospital_observed_48h", "followup_complete"]
    one_sided = [c for c in optional if (c in filtered) != (c in full)]
    if one_sided:
        raise ValueError("Columns present in only one cohort CSV: " + str(one_sided))
    temporal = ["extubation_time", "anchor_year", "anchor_year_group"]
    check_columns = features + [main["target"]] + [c for c in temporal if c in filtered and c in full]
    check_columns += [c for c in optional if c in filtered]
    try:
        pd.testing.assert_frame_equal(filtered[check_columns].reset_index(drop=True),
            shared[check_columns].reset_index(drop=True), check_dtype=False,
            check_exact=False, rtol=1e-8, atol=1e-10)
    except AssertionError:
        raise ValueError("Shared patients have inconsistent features, outcomes or index-event metadata across the two CSVs.") from None
    for cfg in [configs["filtered_six_hour"], configs["full_six_hour"]]:
        cfg["feature_columns"] = features.copy()
    if "filtered_last_hour" in configs:
        last_cfg = configs["filtered_last_hour"]
        last_cfg["feature_columns"] = None
        _, last_nums, last_cats, _, _, _ = load_cohort(last_cfg)
        expected = {feature_family(c) for c in features}
        if expected != {feature_family(c) for c in last_nums + last_cats}:
            raise ValueError("The last-hour sensitivity must use the same variables as the main model.")
        last_cfg["feature_columns"] = last_nums + last_cats
    main_manifest, _, _, _ = make_split(filtered, main)
    full_manifest, _, _, _ = make_split(full, configs["full_six_hour"])
    matching = full_manifest.set_index("subject_id").loc[filtered.subject_id, "partition"].to_numpy()
    if not np.array_equal(main_manifest.partition.to_numpy(), matching):
        raise ValueError("Shared patients must have the same temporal partition in both cohorts.")
    return configs


def environment_versions():
    """Package versions saved with every analysis."""
    versions = {p: importlib.metadata.version(p) for p in
        ["numpy", "pandas", "scikit-learn", "xgboost", "scipy", "matplotlib", "joblib"]}
    return {**versions, "python": platform.python_version()}


def suite_plan(configs):
    """Settings, input hashes and environment that must stay fixed once the holdout is opened."""
    return {"analyses": {name: {"config": cfg, "source_hashes": load_cohort(cfg)[-1]}
        for name, cfg in configs.items()}, "environment": environment_versions()}


def guard_final_suite(configs):
    """Once any holdout is evaluated, all analyses must keep their frozen inputs/models."""
    marker = Path(next(iter(configs.values()))["suite_output_dir"]) / "final_suite_started.json"
    if not marker.exists():
        return
    saved = json.loads(marker.read_text())
    if saved["plan"] != suite_plan(configs):
        raise RuntimeError("Final evaluation has started. Restore the frozen analysis set, settings, data and environment.")
    for name, cfg in configs.items():
        out = Path(cfg["output_dir"])
        if file_hash(out / "trained_models.joblib") != saved["model_hashes"][name]:
            raise RuntimeError("A model changed after final evaluation started.")
        if file_hash(out / "frozen_analysis.json") != saved["analysis_hashes"][name]:
            raise RuntimeError("Frozen analysis metadata changed after final evaluation started.")


In [ ]:
RUN_CONFIGS = make_run_configs(CONFIG)
RUN_CONFIGS = prepare_analyses(RUN_CONFIGS)
guard_final_suite(RUN_CONFIGS)
SUITE_DIR = Path(next(iter(RUN_CONFIGS.values()))["suite_output_dir"])
SUITE_DIR.mkdir(parents=True, exist_ok=True)

# One row per analysis: cohort size, events, holdout feasibility and data-quality counts.
INPUT_CHECKS = pd.DataFrame({name: check_inputs(cfg) for name, cfg in RUN_CONFIGS.items()}).T
INPUT_CHECKS.to_csv(SUITE_DIR / "input_checks.csv")
display(INPUT_CHECKS)

# Error detection for the extraction step: implausible values are counted, not changed.
# The full cohort contains every filtered patient, so one audit covers both.
_full, _full_nums = load_cohort(RUN_CONFIGS["full_six_hour"])[:2]
RANGE_AUDIT = range_audit(_full, _full_nums)
RANGE_AUDIT.to_csv(SUITE_DIR / "predictor_range_audit_full_cohort.csv", index=False)
flagged = RANGE_AUDIT[(RANGE_AUDIT.below + RANGE_AUDIT.above) > 0]
if len(flagged):
    warnings.warn(f"{len(flagged)} predictor columns contain values outside PLAUSIBLE_RANGES; report them to the extraction step.")
    display(flagged.sort_values("implausible_fraction", ascending=False))


## 4. Cohort description

Development patients only: study timeline, Table 1 by outcome and missingness-filter membership, and distributions of selected six-hour means before fold-wise imputation. Averaged inputs do not retain hourly trajectories or variability.


In [ ]:
def development_patients(cfg):
    """Development partition of one cohort, with its predictor list."""
    df, nums, _, _, _, _ = load_cohort(cfg)
    _, dev, _, _ = make_split(df, cfg)
    return df.iloc[dev].reset_index(drop=True), nums


def plot_study_timeline(folder):
    """Timeline figure: predictor window, prediction time and outcome window (hours from extubation)."""
    fig, (before, after) = plt.subplots(1, 2, figsize=(10, 2.6), sharey=True,
        gridspec_kw={"width_ratios": [1, 1.6], "wspace": 0.04})
    before.axvspan(-6, 0, color="tab:blue", alpha=.2)
    before.axvspan(-1, 0, color="tab:blue", alpha=.35)
    before.text(-3.5, .7, "Predictors\nsix-hour means\n[t − 6 h, t)", ha="center", va="center", fontsize=9)
    before.text(-0.5, .2, "last-hour\nsensitivity", ha="center", va="center", fontsize=7)
    before.set(xlim=(-6.3, 0), xticks=range(-6, 1), ylim=(0, 1), yticks=[])
    after.axvspan(0, 48, color="tab:red", alpha=.15)
    after.text(24, .6, "Outcome window (t, t + 48 h]\nreintubation or death", ha="center", va="center", fontsize=9)
    after.text(1, .12, "t = planned extubation: prediction time, no gap", fontsize=8)
    after.set(xlim=(0, 48.5), xticks=[12, 24, 36, 48])  # 0 is labelled on the left panel
    for ax in (before, after):
        ax.axvline(0, color="k", lw=2)
        for side in ["left", "right", "top"]:
            ax.spines[side].set_visible(False)
    fig.supxlabel("Hours relative to planned extubation (t)", fontsize=9, y=-0.06)
    fig.savefig(folder / "study_timeline.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


def describe_group(d, nums, target):
    """Table 1 column for one group of patients."""
    def pct(mask):
        return f"{int(mask.sum())} ({100 * mask.mean():.1f}%)" if len(d) else ""
    row = {"Patients, n": str(len(d))}
    age_col = next((c for c in ["admission_age", "age"] if c in d), None)
    if age_col and len(d):
        q = d[age_col].quantile([.25, .5, .75])
        row["Age, median [IQR]"] = f"{q[.5]:.0f} [{q[.25]:.0f}–{q[.75]:.0f}]"
    sex_col = next((c for c in ["gender", "sex"] if c in d), None)
    if sex_col:
        row["Female, n (%)"] = pct(d[sex_col].astype(str).str.upper().str.startswith("F"))
    if "race" in d:
        groups = pd.Series(race_group(d["race"]), index=d.index)
        for g in ["White", "Black", "Hispanic/Latino", "Asian", "Other/unknown"]:
            row[f"Race: {g}, n (%)"] = pct(groups.eq(g))
    row["Extubation failure, n (%)"] = pct(d[target].eq(1))
    for c, label in [("death_within_48_hours", "  Death within 48 h"),
                     ("reintubation_within_48_hours", "  Reintubation within 48 h")]:
        if c in d:
            row[label + ", n (%)"] = pct(d[c].eq(1))
    followup_col = next((c for c in ["hospital_observed_48h", "followup_complete"] if c in d), None)
    if followup_col:
        row["Incomplete 48-h hospital observation, n (%)"] = pct(d[followup_col].ne(1))
    row["Predictor values missing, mean %"] = f"{100 * d[nums].isna().to_numpy().mean():.1f}" if len(d) else ""
    return row


def cohort_characteristics(configs, folder):
    """Table 1: who the missingness filter keeps and removes (development patients)."""
    main_cfg = configs["filtered_six_hour"]
    target = main_cfg["target"]
    main, nums = development_patients(main_cfg)
    full, _ = development_patients(configs["full_six_hour"])
    removed = full.loc[~full.subject_id.isin(main.subject_id)]
    groups = {"Main cohort: no failure": main[main[target].eq(0)],
              "Main cohort: failure": main[main[target].eq(1)],
              "Main cohort: all": main,
              "Removed by missingness filter": removed,
              "Full cohort: all": full}
    table = pd.DataFrame({name: describe_group(d, nums, target) for name, d in groups.items()})
    table.to_csv(folder / "table1_development_patients.csv")
    return table


def plot_mean_distributions(d, nums, target, path, title):
    """Distributions of supplied six-hour means in development patients."""
    families = ["heart_rate", "resp_rate", "spo2", "rsbi"]
    columns = [c for family in families for c in nums if feature_family(c) == family]
    if not columns:
        return
    fig, axes = plt.subplots(1, len(columns), figsize=(4 * len(columns), 3), squeeze=False)
    for ax, c in zip(axes[0], columns):
        values = d[c].dropna()
        if values.empty:
            continue
        low, high = values.quantile([.01, .99])
        if low == high:
            low, high = low - .5, high + .5
        for outcome, label in [(0, "No failure"), (1, "Failure")]:
            v = d.loc[d[target].eq(outcome), c].dropna()
            if len(v):
                ax.hist(v, bins=25, range=(low, high), density=True, histtype="step", label=label)
        ax.set(xlabel=c, ylabel="Density within displayed range")
        ax.legend(fontsize=8)
    fig.suptitle(title + " (1st–99th percentiles; non-missing values)")
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)


In [ ]:
plot_study_timeline(SUITE_DIR)
TABLE_1 = cohort_characteristics(RUN_CONFIGS, SUITE_DIR)
display(TABLE_1)
for name in ["filtered_six_hour", "full_six_hour"]:
    patients, predictors = development_patients(RUN_CONFIGS[name])
    plot_mean_distributions(patients, predictors, RUN_CONFIGS[name]["target"],
        SUITE_DIR / f"{name}_mean_distributions.png", f"{name}: development patients")
display(Image(filename=str(SUITE_DIR / "study_timeline.png")))
display(Image(filename=str(SUITE_DIR / "filtered_six_hour_mean_distributions.png")))


## 5. Models, tuning and calibration

- **Models.** Logistic regression (LR, interpretable, linear effects) and XGBoost (black box, nonlinear effects and interactions).
- **Preprocessing.** Median imputation, plus standardisation for LR, is fitted inside each training fold, so no information from validation or test patients leaks in. Predictors are otherwise used as supplied: no missingness indicators or engineered features are added.
- **Tuning.** An exhaustive `GridSearchCV` runs in the inner folds and refits on AUROC. Average precision and log loss are recorded for each candidate. LR searches `C` over 0.001–100; XGBoost searches all 48 combinations in `xgb_param_grid`.
- **Class imbalance.** LR uses balanced class weights; XGBoost uses the training-fold negative/positive ratio (`scale_pos_weight`).
- **Calibration.** Class weighting inflates predicted risks, so each tuned pipeline is recalibrated with sigmoid (Platt) scaling, using cross-validated predictions within the training partition.

In [ ]:
class FoldBalancedXGB(ClassifierMixin, BaseEstimator):
    """XGBoost whose scale_pos_weight is recalculated from each training fold."""
    def __init__(self, n_estimators=250, max_depth=3, learning_rate=0.05,
                 min_child_weight=1, reg_lambda=1, subsample=0.8,
                 colsample_bytree=0.8, random_state=42, n_jobs=4):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.min_child_weight = min_child_weight
        self.reg_lambda = reg_lambda
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.random_state = random_state
        self.n_jobs = n_jobs

    def fit(self, X, y):
        counts = np.bincount(np.asarray(y, dtype=int), minlength=2)
        if min(counts) == 0:
            raise ValueError("XGBoost fit requires both classes.")
        # Negative/positive ratio of this training fold only.
        self.scale_pos_weight_ = float(counts[0] / counts[1])
        self.model_ = XGBClassifier(**self.get_params(), scale_pos_weight=self.scale_pos_weight_,
            objective="binary:logistic", eval_metric="logloss", tree_method="hist", device="cpu")
        self.model_.fit(X, y)
        self.classes_ = self.model_.classes_
        self.n_features_in_ = self.model_.n_features_in_
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    def predict(self, X):
        return self.model_.predict(X)


def model_pipeline(name, nums, cats, cfg):
    """Fold-wise preprocessing and model. Entirely missing training columns are kept (imputed as 0)."""
    numeric = [("impute", SimpleImputer(strategy="median", keep_empty_features=True))]
    if name == "logistic_regression":
        numeric.append(("scale", StandardScaler()))
    parts = [("numeric", Pipeline(numeric), nums)]
    if cats:
        categorical = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown", keep_empty_features=True)),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
        parts.append(("categorical", categorical, cats))
    prep = ColumnTransformer(parts, remainder="drop", sparse_threshold=0)
    if name == "logistic_regression":
        model = LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=cfg["lr_max_iter"], random_state=cfg["seed"])
    elif name == "xgboost":
        model = FoldBalancedXGB(random_state=cfg["seed"], n_jobs=cfg["xgb_threads"],
            subsample=cfg["xgb_subsample"], colsample_bytree=cfg["xgb_colsample_bytree"])
    else:
        raise ValueError(name)
    return Pipeline([("preprocess", prep), ("model", model)])


def fit_tuned(name, X, y, nums, cats, cfg, cv):
    """Exhaustive grid search on the given folds (refit on AUROC), then optional calibration."""
    pipe = model_pipeline(name, nums, cats, cfg)
    grid = ({"model__C": cfg["lr_c_values"]} if name == "logistic_regression"
            else {"model__" + key: values for key, values in cfg["xgb_param_grid"].items()})
    search = GridSearchCV(pipe, grid, cv=cv, refit="auroc", n_jobs=1, error_score="raise",
        scoring={"auroc": "roc_auc", "average_precision": "average_precision", "log_loss": "neg_log_loss"})
    search.fit(X, y)
    fitted = search.best_estimator_
    if cfg["calibrate"]:
        # Calibrate on cross-validated predictions within this training partition, then refit on all of it.
        fitted = CalibratedClassifierCV(clone(fitted), method=cfg["calibration_method"], cv=cv, ensemble=False, n_jobs=1)
        fitted.fit(X, y)
    cols = [c for c in search.cv_results_ if c == "params" or c.startswith(("mean_test_", "std_test_", "rank_test_", "mean_fit_"))]
    return fitted, search.best_params_, pd.DataFrame(search.cv_results_)[cols]

## 6. Metrics, thresholds and figures

- **Discrimination:** AUROC and average precision (AP, the step-based area under the precision–recall curve; compare it with prevalence).
- **Accuracy of predicted risks:** Brier score; scaled Brier = 1 − Brier / (prevalence × (1 − prevalence)), where 0 means no better than predicting the prevalence.
- **Calibration:** observed:expected (O:E) ratio, where 1 is ideal, and calibration slope (logistic regression of the outcome on logit risk), where 1 is ideal and below 1 indicates overfitting.
- **Classification at the chosen threshold:** sensitivity, specificity, PPV and NPV.
- **Baselines:** prevalence and a fixed-seed random score. RSBI summary (exploratory threshold 105; an averaged value is not the original bedside measurement) is compared on the same RSBI-available patients.
- **Threshold.** For each model, use the development out-of-fold (OOF) threshold with maximum specificity among those reaching ≥ 80% sensitivity. This exploratory operating point prioritises catching failures; its metrics on development data are descriptive. Thresholds are frozen before the test evaluation.
- **Clinical utility.** A decision curve gives the net benefit of acting on model alerts across risk thresholds of 1–30%, compared with acting on everyone, on no one, or on RSBI ≥ 105.
- **Uncertainty.** Patient bootstrap 95% intervals. Paired bootstraps use the same resampled patients for both compared models.
- **Subgroups.** Exploratory checks by sex, age group (< 65 or ≥ 65) and race group. Age and sex are also predictors; race, when available, is used only for evaluation.


In [ ]:
def safe_ratio(a, b):
    return float(a / b) if b else np.nan


def calibration_summary(y, p):
    """Observed:expected ratio and calibration slope (outcome regressed on logit risk)."""
    p = np.clip(p, 1e-7, 1 - 1e-7)
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    oe = safe_ratio(y.sum(), p.sum())
    if len(np.unique(y)) < 2 or np.std(logit) < 1e-12:
        return oe, np.nan
    # A very large C makes this effectively an unpenalised logistic regression.
    slope = LogisticRegression(C=1e10, max_iter=1000).fit(logit, y).coef_[0, 0]
    return oe, float(slope)


def metrics(y, score, threshold=0.5, probability=True, calibration=True):
    """Discrimination, threshold and (for probabilities) calibration metrics."""
    y, score = np.asarray(y, int), np.asarray(score, float)
    keep = np.isfinite(score)
    y, score = y[keep], score[keep]
    if not len(y):
        return {"n": 0}
    two = len(np.unique(y)) == 2
    prevalence = float(y.mean())
    tn, fp, fn, tp = confusion_matrix(y, score >= threshold, labels=[0, 1]).ravel()
    result = {"n": len(y), "events": int(y.sum()), "prevalence": prevalence,
        "auroc": roc_auc_score(y, score) if two else np.nan,
        "average_precision": average_precision_score(y, score) if two else np.nan,
        "threshold": float(threshold), "sensitivity": safe_ratio(tp, tp + fn),
        "specificity": safe_ratio(tn, tn + fp), "ppv": safe_ratio(tp, tp + fp),
        "npv": safe_ratio(tn, tn + fn), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "brier": np.nan, "scaled_brier": np.nan, "log_loss": np.nan,
        "observed_expected_ratio": np.nan, "calibration_slope": np.nan}
    if probability:
        result["brier"] = brier_score_loss(y, score)
        result["scaled_brier"] = 1 - result["brier"] / (prevalence * (1 - prevalence)) if two else np.nan
        result["log_loss"] = log_loss(y, np.clip(score, 1e-7, 1 - 1e-7), labels=[0, 1])
        if calibration:
            result["observed_expected_ratio"], result["calibration_slope"] = calibration_summary(y, score)
    return result


def choose_threshold(y, score, sensitivity_target):
    """Maximise specificity at the target sensitivity using development OOF scores."""
    if not 0 < sensitivity_target <= 1:
        raise ValueError("sensitivity_target must be in (0,1].")
    fpr, tpr, thresholds = roc_curve(y, score, drop_intermediate=False)
    eligible = np.flatnonzero((tpr >= sensitivity_target) & np.isfinite(thresholds))
    # roc_curve thresholds descend: among tied FPR choose the highest threshold.
    best = eligible[np.argmin(fpr[eligible])]
    return float(thresholds[best])


def prediction_table(df, cfg, scores, rsbi_col, fold=None):
    """One row per patient: labels, evaluation-only descriptors and every model's score."""
    cols = [c for c in ["subject_id", "hadm_id", "stay_id", cfg["target"], *OUTCOME_COMPONENTS,
        "age", "admission_age", "gender", "sex", "race"] if c in df]
    out = df[cols].copy().reset_index(drop=True)
    out = out.rename(columns={cfg["target"]: "y_true"})
    for name, values in scores.items():
        out[name] = values
    if rsbi_col:
        out["rsbi"] = df[rsbi_col].to_numpy()
    if fold is not None:
        out["outer_fold"] = fold
    return out


def comparison_tables(predictions, thresholds, calibration=True):
    """Metrics for every model on all patients and on the RSBI-available subset."""
    rows = []
    scopes = {"all_patients": np.ones(len(predictions), dtype=bool)}
    if "rsbi" in predictions:
        scopes["rsbi_available_same_patients"] = np.isfinite(predictions.rsbi)
    for scope, mask in scopes.items():
        p = predictions.loc[mask]
        for name in PROBABILITY_MODELS + ["random_guess"]:
            if name in p:
                # Calibration metrics apply to probability models only.
                rows.append({"scope": scope, "model": name, **metrics(p.y_true, p[name],
                    thresholds.get(name, 0.5), name in PROBABILITY_MODELS, calibration)})
        if scope == "rsbi_available_same_patients":
            rows.append({"scope": scope, "model": "rsbi", **metrics(p.y_true, p.rsbi, 105, False)})
    return pd.DataFrame(rows)


def net_benefit(y, flagged, threshold):
    """Net benefit of acting on flagged patients at one risk threshold (Vickers & Elkin 2006)."""
    n = len(y)
    tp = np.sum(flagged & (y == 1))
    fp = np.sum(flagged & (y == 0))
    return tp / n - fp / n * threshold / (1 - threshold)


def decision_curve(p, max_threshold, include_rsbi=False):
    """Net benefit of each model, treat-all, treat-none and (optionally) RSBI >= 105."""
    y = p.y_true.to_numpy()
    rows = []
    for t in np.round(np.arange(0.01, max_threshold + 1e-9, 0.005), 3):
        strategies = {name: p[name].to_numpy() >= t for name in MODEL_NAMES}
        strategies["treat_all"] = np.ones(len(y), dtype=bool)
        if include_rsbi:
            strategies["rsbi_ge_105"] = p.rsbi.to_numpy() >= 105
        rows += [{"threshold": t, "strategy": k, "net_benefit": net_benefit(y, v, t)} for k, v in strategies.items()]
        rows.append({"threshold": t, "strategy": "treat_none", "net_benefit": 0.0})
    return pd.DataFrame(rows)


def plot_comparison(p, folder, prefix, max_threshold, include_rsbi=False):
    """ROC, precision-recall, calibration and decision curves for one set of predictions."""
    folder = Path(folder)
    if p.empty:
        return
    two_classes = p.y_true.nunique() == 2
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    roc_ax, pr_ax, cal_ax, dca_ax = axes.flat
    prevalence = p.y_true.mean()
    if two_classes:
        for name in MODEL_NAMES + (["rsbi"] if include_rsbi else []):
            fpr, tpr, _ = roc_curve(p.y_true, p[name])
            precision, recall, _ = precision_recall_curve(p.y_true, p[name])
            roc_ax.plot(fpr, tpr, label=f"{name} (AUROC {roc_auc_score(p.y_true, p[name]):.3f})")
            # Step interpolation matches average precision.
            pr_ax.step(recall, precision, where="post", label=f"{name} (AP {average_precision_score(p.y_true, p[name]):.3f})")
        roc_ax.plot([0, 1], [0, 1], "k--", alpha=.5, label="chance")
        pr_ax.axhline(prevalence, color="k", linestyle="--", label=f"prevalence ({prevalence:.3f})")
    else:
        for ax in [roc_ax, pr_ax]:
            ax.text(.5, .5, "Requires both outcome classes", ha="center", va="center", transform=ax.transAxes)
    roc_ax.set(xlabel="1 − specificity", ylabel="Sensitivity", title="ROC curve", xlim=(0, 1), ylim=(0, 1))
    pr_ax.set(xlabel="Recall (sensitivity)", ylabel="Precision (PPV)", title="Precision–recall curve", xlim=(0, 1), ylim=(0, 1))

    # Quantile bins with about ten or more events each, between 3 and 10 bins.
    n_bins = int(np.clip(p.y_true.sum() // 10, 3, 10))
    points = []
    for name in MODEL_NAMES:
        observed, predicted = calibration_curve(p.y_true, p[name], pos_label=1, n_bins=n_bins, strategy="quantile")
        cal_ax.plot(predicted, observed, "o-", label=name)
        points += [{"model": name, "mean_predicted_probability": x, "observed_event_rate": o}
                   for x, o in zip(predicted, observed)]
    pd.DataFrame(points).to_csv(folder / f"{prefix}_calibration_points.csv", index=False)
    limit = min(1, max(.1, 1.15 * max(max(pt["mean_predicted_probability"], pt["observed_event_rate"]) for pt in points)))
    cal_ax.plot([0, limit], [0, limit], "k--", alpha=.5, label="ideal")
    # Distribution of predicted risks behind the calibration curve.
    hist_ax = cal_ax.twinx()
    hist_ax.hist([p[name].clip(upper=limit) for name in MODEL_NAMES], bins=30, range=(0, limit), alpha=.25, color=["tab:blue", "tab:orange"])
    hist_ax.set_ylabel("Patients")
    cal_ax.set_zorder(hist_ax.get_zorder() + 1)
    cal_ax.patch.set_visible(False)
    cal_ax.set(xlabel="Mean predicted risk", ylabel="Observed event rate",
        title=f"Calibration ({n_bins} quantile bins) and risk distribution", xlim=(0, limit), ylim=(0, limit))

    curve = decision_curve(p, max_threshold, include_rsbi)
    curve.to_csv(folder / f"{prefix}_decision_curve.csv", index=False)
    for strategy, g in curve.groupby("strategy", sort=False):
        dca_ax.plot(g.threshold, g.net_benefit, label=strategy,
            linestyle="--" if strategy.startswith("treat") else "-", color="k" if strategy == "treat_none" else None)
    dca_limits = (-0.5 * prevalence, 1.1 * prevalence) if prevalence else (min(-.01, curve.net_benefit.min()), .01)
    dca_ax.set(xlabel="Risk threshold for acting on an alert", ylabel="Net benefit", title="Decision curve",
        xlim=(0, max_threshold), ylim=dca_limits)
    for ax in axes.flat:
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=7)
    fig.suptitle(f"{prefix}: {len(p)} patients, {int(p.y_true.sum())} events")
    fig.tight_layout()
    fig.savefig(folder / f"{prefix}_curves.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


def bootstrap_intervals(p, thresholds, cfg):
    """Patient bootstrap; retain each metric's finite replicates, including single-class draws."""
    rng = np.random.default_rng(cfg["seed"] + 200)
    samples, differences = [], []
    for rep in range(cfg["bootstrap_replicates"]):
        sample = p.iloc[rng.integers(0, len(p), len(p))]
        table = comparison_tables(sample, thresholds, calibration=False)
        for _, row in table.iterrows():
            for metric in ["auroc", "average_precision", "sensitivity", "specificity", "ppv", "npv", "brier", "scaled_brier"]:
                samples.append({"replicate": rep, "scope": row.scope, "model": row.model, "metric": metric, "value": row.get(metric, np.nan)})
        paired = table[table.scope.eq("all_patients")].set_index("model")
        for metric in ["auroc", "average_precision", "brier"]:
            differences.append({"replicate": rep, "metric": metric, "value": paired.loc["xgboost", metric] - paired.loc["logistic_regression", metric]})
    def summarise(frame, groups):
        if frame.empty:
            return pd.DataFrame(columns=groups + ["valid_replicates", "lower_95", "upper_95"])
        frame["value"] = frame["value"].where(np.isfinite(frame["value"]))
        return frame.groupby(groups).value.agg(valid_replicates="count", lower_95=lambda s: s.quantile(.025), upper_95=lambda s: s.quantile(.975)).reset_index()
    return summarise(pd.DataFrame(samples), ["scope", "model", "metric"]), summarise(pd.DataFrame(differences), ["metric"])


def paired_bootstrap(y, a, b, reps, seed):
    """Metrics of scores a and b and their difference (a - b) on the same resampled patients."""
    y, a, b = np.asarray(y, int), np.asarray(a, float), np.asarray(b, float)
    if not len(y):
        return []
    def values(i):
        two_classes = len(np.unique(y[i])) == 2
        return {"auroc": (roc_auc_score(y[i], a[i]), roc_auc_score(y[i], b[i])) if two_classes else (np.nan, np.nan),
                "average_precision": (average_precision_score(y[i], a[i]), average_precision_score(y[i], b[i])) if two_classes else (np.nan, np.nan),
                "brier": (brier_score_loss(y[i], a[i]), brier_score_loss(y[i], b[i]))}
    point = values(np.arange(len(y)))
    rng = np.random.default_rng(seed)
    draws = [values(rng.integers(0, len(y), len(y))) for _ in range(reps)]
    rows = []
    for metric, (va, vb) in point.items():
        diff = np.array([d[metric][0] - d[metric][1] for d in draws])
        diff = diff[np.isfinite(diff)]
        rows.append({"metric": metric, "a": va, "b": vb, "a_minus_b": va - vb,
            "lower_95": np.quantile(diff, .025) if len(diff) else np.nan,
            "upper_95": np.quantile(diff, .975) if len(diff) else np.nan, "valid_replicates": len(diff)})
    return rows


def outcome_component_auroc(p):
    """AUROC for deaths and reintubations separately, each against patients with neither event."""
    rows = []
    negatives = p.y_true.eq(0)
    for component in OUTCOME_COMPONENTS:
        if component not in p:
            continue
        for name in MODEL_NAMES + (["rsbi"] if "rsbi" in p else []):
            keep = (negatives | p[component].eq(1)) & np.isfinite(p[name])
            y = p.loc[keep, component]
            rows.append({"component": component, "model": name, "events": int(y.sum()), "non_events": int((y == 0).sum()),
                "auroc": roc_auc_score(y, p.loc[keep, name]) if y.nunique() == 2 else np.nan})
    return pd.DataFrame(rows)


def bootstrap_auroc_ci(y, s, reps, seed):
    """Percentile bootstrap interval for one AUROC."""
    y, s = np.asarray(y, int), np.asarray(s, float)
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(reps):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            values.append(roc_auc_score(y[i], s[i]))
    return (np.quantile(values, .025), np.quantile(values, .975)) if values else (np.nan, np.nan)


def subgroup_metrics(p, thresholds, cfg):
    """Exploratory performance by sex, age group and race group (descriptive subgroup checks)."""
    groups = pd.DataFrame(index=p.index)
    sex_col = next((c for c in ["gender", "sex"] if c in p), None)
    if sex_col:
        groups["sex"] = p[sex_col].astype(str)
    age_col = next((c for c in ["admission_age", "age"] if c in p), None)
    if age_col:
        groups["age_group"] = np.where(p[age_col] >= 65, ">=65", "<65")
    if "race" in p:
        groups["race_group"] = race_group(p["race"])
    rows = []
    for variable in groups:
        for value, idx in groups.groupby(variable).groups.items():
            g = p.loc[idx]
            for name in MODEL_NAMES:
                row = {"variable": variable, "group": value, "model": name,
                    **metrics(g.y_true, g[name], thresholds[name], calibration=False)}
                row["auroc_lower_95"], row["auroc_upper_95"] = bootstrap_auroc_ci(
                    g.y_true, g[name], min(cfg["bootstrap_replicates"], 500), cfg["seed"] + 500)
                rows.append(row)
    table = pd.DataFrame(rows)
    if len(table):
        table["interpretation"] = "exploratory; small subgroups are unstable"
    return table


def plot_forest(labels, points, lower, upper, path, title, xlabel="AUROC (95% bootstrap CI)", series=None):
    """Horizontal point-and-interval plot; series (optional) offsets several models per row."""
    keys = list(dict.fromkeys(labels))
    series = series if series is not None else ["estimate"] * len(labels)
    names = list(dict.fromkeys(series))
    offsets = np.linspace(-.15, .15, len(names)) if len(names) > 1 else [0]
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(keys) + 1.5))
    for name, offset in zip(names, offsets):
        idx = [i for i, s in enumerate(series) if s == name]
        y = [keys.index(labels[i]) + offset for i in idx]
        x = np.array([points[i] for i in idx], dtype=float)
        lo = np.array([lower[i] for i in idx], dtype=float)
        hi = np.array([upper[i] for i in idx], dtype=float)
        line, = ax.plot(x, y, "o", label=name if len(names) > 1 else None)
        # A percentile interval need not contain its point estimate.
        valid = np.isfinite(lo) & np.isfinite(hi)
        yy = np.asarray(y)[valid]
        ax.hlines(yy, lo[valid], hi[valid], color=line.get_color())
        ax.vlines(lo[valid], yy - .06, yy + .06, color=line.get_color())
        ax.vlines(hi[valid], yy - .06, yy + .06, color=line.get_color())
    ax.set_yticks(range(len(keys)), keys, fontsize=8)
    ax.axvline(.5, color="k", linestyle="--", alpha=.5)
    ax.invert_yaxis()
    ax.set(xlabel=xlabel, title=title)
    if len(names) > 1:
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def plot_subgroups(table, path, title):
    """Forest plot of subgroup AUROC for both models."""
    t = table.dropna(subset=["auroc"]) if len(table) else table
    if t.empty:
        return
    labels = [f"{v}: {g} (n={n}, events={e})" for v, g, n, e in zip(t.variable, t.group, t.n, t.events)]
    plot_forest(labels, t.auroc.tolist(), t.auroc_lower_95.tolist(), t.auroc_upper_95.tolist(), path, title, series=t.model.tolist())


## 7. Development results and sensitivity analyses

Nested CV compares LR and XGBoost by mean outer-fold AUROC. Thresholds come from development OOF predictions. Both models are then retuned on all development patients. Settings, features, thresholds, source hashes, package versions and fitted models are saved and reused on identical reruns.

Two sensitivity analyses:
1. **Missingness filter:** retrain on the full cohort. Compare its overall results, paired OOF results on shared patients, and performance on patients excluded from the main cohort. Population, training size and fold composition differ, so these comparisons are exploratory.
2. **Feature representation:** compare six-hour means with the last available hourly summary on identical main-cohort patients and folds. Bin 5 includes forward filling; hourly RSBI also differs in its ratio calculation. This tests the supplied representations, not a pure time-window effect or a trend/variability hypothesis. Set `include_last_hour_ablation=False` to omit.

Additional descriptive checks separate deaths and reintubations and examine subgroups. Patient-bootstrap intervals for pooled OOF results do not include uncertainty from repeating model development. They are not final test results.


In [ ]:
def run_development(config):
    """Nested CV, thresholds, development figures and final development refit for one analysis."""
    cfg = validate_config(config)
    out = Path(cfg["output_dir"])
    df, nums, cats, rsbi_col, audit, hashes = load_cohort(cfg)
    if (out / "frozen_analysis.json").exists():
        saved_mode = json.loads((out / "frozen_analysis.json").read_text())["config"].get("run_mode", "main")
        if saved_mode != cfg["run_mode"]:
            raise ValueError("This output directory belongs to a different run mode; use separate output folders for each mode.")
    locked = (out / "final_test_started.json").exists() or (Path(cfg["suite_output_dir"]) / "final_suite_started.json").exists()
    cache_files = ["frozen_analysis.json", "trained_models.joblib", "development_oof_predictions.csv",
        "patient_split_manifest.csv", "environment_versions.json", "threshold_derivation_DESCRIPTIVE_NOT_VALIDATION.csv",
        "development_oof_bootstrap_CI.csv"]
    cached = all((out / name).exists() for name in cache_files)
    reuse = False
    if cached or locked:
        frozen = json.loads((out / "frozen_analysis.json").read_text())
        reuse = (cached and cfg == frozen["config"] and hashes == frozen["source_hashes"]
            and json.loads((out / "environment_versions.json").read_text()) == environment_versions())
        if locked and not reuse:
            raise RuntimeError("Final test already opened and settings/data/environment changed. Restore the frozen analysis.")
    if reuse:
        bundle = joblib.load(out / "trained_models.joblib")
        dev = df.set_index("subject_id", drop=False).loc[bundle["development_ids"]].reset_index(drop=True)
        print("Reusing completed development: settings, source data and environment are unchanged.")
        return {"bundle": bundle, "data": df, "development": dev,
            "oof": pd.read_csv(out / "development_oof_predictions.csv", dtype={"subject_id": "string"}),
            "manifest": pd.read_csv(out / "patient_split_manifest.csv", dtype={"subject_id": "string"}), "output_dir": out}
    manifest, dev_idx, test_idx, split_note = make_split(df, cfg)
    holdout = holdout_status(df, test_idx, cfg)
    out.mkdir(parents=True, exist_ok=True)
    dev = df.iloc[dev_idx].reset_index(drop=True)
    followup = followup_summary(dev, cfg)
    write_json(out / "development_followup.json", followup)
    if followup["negative_labels_without_complete_observation"]:
        warnings.warn(f"{cfg['analysis']}: {followup['negative_labels_without_complete_observation']} development negatives lack complete 48-hour observation. Supplied labels are retained; describe this limitation.")
    X, y = dev[nums + cats], dev[cfg["target"]].to_numpy()
    if min(np.bincount(y, minlength=2)) < 50:
        warnings.warn("Fewer than 50 development patients in one class; estimates may be unstable.")

    # Outer loop: each fold's patients are scored by models tuned without them.
    outer = checked_cv(y, cfg["outer_folds"], cfg["seed"])
    scores = {n: np.full(len(dev), np.nan) for n in PROBABILITY_MODELS}
    # Fixed random scores provide a reproducible chance baseline.
    scores["random_guess"] = np.random.default_rng(cfg["seed"]).uniform(size=len(dev))
    folds = np.full(len(dev), -1, dtype=int)
    fold_results, best_params = [], []
    for fold, (train, valid) in enumerate(outer):
        inner = checked_cv(y[train], cfg["inner_folds"], cfg["seed"] + fold + 1)
        folds[valid] = fold
        scores["prevalence_baseline"][valid] = y[train].mean()
        for name in MODEL_NAMES:
            print(f"Outer fold {fold + 1}/{len(outer)}: {name}", flush=True)
            fitted, params, search_table = fit_tuned(name, X.iloc[train], y[train], nums, cats, cfg, inner)
            scores[name][valid] = fitted.predict_proba(X.iloc[valid])[:, 1]
            search_table.to_csv(out / f"search_outer_{fold}_{name}.csv", index=False)
            best_params.append({"outer_fold": fold, "model": name, "parameters": params})
        for name, values in scores.items():
            fold_results.append({"outer_fold": fold, "model": name, **metrics(y[valid], values[valid], 0.5, name in PROBABILITY_MODELS)})
    assert all(np.isfinite(s).all() for s in scores.values()) and (folds >= 0).all()
    oof = prediction_table(dev, cfg, scores, rsbi_col, folds)
    oof.to_csv(out / "development_oof_predictions.csv", index=False)
    manifest["outer_fold"] = -1
    manifest.loc[dev_idx, "outer_fold"] = folds
    manifest.to_csv(out / "patient_split_manifest.csv", index=False)

    # Fold-level results and model selection by mean outer-fold AUROC.
    fold_frame = pd.DataFrame(fold_results)
    fold_frame.to_csv(out / "outer_fold_metrics_fixed_0p5.csv", index=False)
    comparison_tables(oof, {}).to_csv(out / "development_metrics_fixed_0p5.csv", index=False)
    fold_frame.groupby("model")[["auroc", "average_precision", "brier", "scaled_brier", "calibration_slope"]].agg(["mean", "std"]).to_csv(out / "outer_fold_summary_mean_sd.csv")
    paired_folds = fold_frame[fold_frame.model.isin(MODEL_NAMES)].pivot(index="outer_fold", columns="model", values="auroc")
    paired_folds["xgboost_minus_logistic_regression"] = paired_folds.xgboost - paired_folds.logistic_regression
    paired_folds.to_csv(out / "outer_fold_paired_auroc_difference.csv")
    ranked = fold_frame[fold_frame.model.isin(MODEL_NAMES)].groupby("model").auroc.mean().sort_values(ascending=False)
    winner = str(ranked.index[0])

    # Thresholds from development OOF scores; their metrics here are descriptive.
    thresholds = {name: choose_threshold(y, scores[name], cfg["sensitivity_target"]) for name in MODEL_NAMES}
    thresholds.update({"prevalence_baseline": 0.5, "random_guess": 0.5})
    comparison_tables(oof, thresholds).to_csv(out / "threshold_derivation_DESCRIPTIVE_NOT_VALIDATION.csv", index=False)
    ci, differences = bootstrap_intervals(oof, thresholds, cfg)
    ci.to_csv(out / "development_oof_bootstrap_CI.csv", index=False)
    differences.to_csv(out / "development_paired_difference_XGB_minus_LR_CI.csv", index=False)

    # Figures and secondary development analyses.
    plot_comparison(oof, out, "development_oof", cfg["decision_curve_max_threshold"])
    if "rsbi" in oof and oof.loc[np.isfinite(oof.rsbi), "y_true"].nunique() == 2:
        plot_comparison(oof.loc[np.isfinite(oof.rsbi)], out, "development_rsbi_matched_subset",
            cfg["decision_curve_max_threshold"], include_rsbi=True)
    outcome_component_auroc(oof).to_csv(out / "development_outcome_component_auroc.csv", index=False)
    subgroups = subgroup_metrics(oof, thresholds, cfg)
    subgroups.to_csv(out / "development_subgroup_metrics.csv", index=False)
    plot_subgroups(subgroups, out / "development_subgroup_auroc.png", f"{cfg['analysis']}: development OOF subgroups (exploratory)")
    range_audit(dev, nums).to_csv(out / "development_predictor_range_audit.csv", index=False)
    dev[nums + cats].isna().mean().rename("missing_fraction_development").to_csv(out / "development_feature_missingness.csv")
    audit.to_csv(out / "feature_audit.csv", index=False)

    # Final development models: retuned on all development patients, used for the holdout.
    final_models, final_params = {}, {}
    final_cv = checked_cv(y, cfg["inner_folds"], cfg["seed"] + 100)
    for name in MODEL_NAMES:
        print(f"Refitting {name} on development patients only", flush=True)
        fitted, params, search_table = fit_tuned(name, X, y, nums, cats, cfg, final_cv)
        final_models[name] = fitted
        final_params[name] = params
        search_table.to_csv(out / f"search_final_development_{name}.csv", index=False)
    write_json(out / "outer_best_parameters.json", best_params)
    versions = environment_versions()
    write_json(out / "environment_versions.json", versions)
    frozen = {"config": cfg, "source_hashes": hashes, "split": split_note,
        "selected_features": nums + cats, "rsbi_column": rsbi_col,
        "partition_counts": manifest.partition.value_counts().to_dict(), "holdout": holdout,
        "thresholds": thresholds, "threshold_rule": "maximum specificity at development OOF sensitivity >= configured target; exploratory, not clinically validated",
        "selected_primary_model": winner, "selection_rule": "highest mean outer-fold AUROC, before final test",
        "mean_outer_fold_auroc": ranked.to_dict(), "final_parameters": final_params,
        "development_n": len(dev), "development_events": int(y.sum()),
        "development_followup": followup,
        "test_outcomes_used_for_training_or_selection": False}
    write_json(out / "frozen_analysis.json", frozen)
    bundle = {"models": final_models, "frozen": frozen, "development_prevalence": float(y.mean()),
        "development_ids": dev.subject_id.tolist(), "test_ids": df.iloc[test_idx].subject_id.tolist()}
    joblib.dump(bundle, out / "trained_models.joblib")
    print(f"{cfg['run_mode'].capitalize()} development complete. Selected model: {winner}. Final test not evaluated.")
    return {"bundle": bundle, "data": df, "development": dev, "oof": oof, "manifest": manifest, "output_dir": out}


def export_analysis_comparison(runs, final=False):
    """One table across analyses, with patient-bootstrap intervals for AUROC and AP."""
    rows = []
    for name, run in runs.items():
        frozen = run["bundle"]["frozen"]
        file = "FINAL_test_metrics.csv" if final else "threshold_derivation_DESCRIPTIVE_NOT_VALIDATION.csv"
        table = pd.read_csv(run["output_dir"] / file)
        table = table.loc[table.scope.eq("all_patients")].copy()
        ci_file = run["output_dir"] / ("FINAL_test_patient_bootstrap_CI.csv" if final else "development_oof_bootstrap_CI.csv")
        if ci_file.exists():
            ci = pd.read_csv(ci_file)
            ci = ci[ci.scope.eq("all_patients") & ci.metric.isin(["auroc", "average_precision"])]
            wide = ci.pivot(index="model", columns="metric", values=["lower_95", "upper_95"])
            wide.columns = [f"{metric}_{bound}" for bound, metric in wide.columns]
            table = table.merge(wide, left_on="model", right_index=True, how="left")
        for col in ["auroc_lower_95", "auroc_upper_95", "average_precision_lower_95", "average_precision_upper_95"]:
            if col not in table:
                table[col] = np.nan
        table.insert(0, "analysis", name)
        table["feature_set"] = frozen["config"]["feature_set"]
        table["predictors"] = len(frozen["selected_features"])
        table["selected_primary_model"] = frozen["selected_primary_model"]
        table["development_patients"] = frozen["development_n"]
        table["development_prevalence"] = frozen["development_events"] / frozen["development_n"]
        table["test_patients"] = frozen["holdout"]["test_patients"]
        table["test_events"] = frozen["holdout"]["test_events"]
        table["small_holdout_warning"] = frozen["holdout"]["small_sample_warning"]
        table["excluded_ambiguous_year"] = frozen["partition_counts"].get("ambiguous_time", 0)
        table["temporal_cutoff"] = frozen["split"]["cutoff"]
        table["mean_outer_fold_auroc"] = table.model.map(frozen["mean_outer_fold_auroc"])
        table["evaluation"] = "frozen temporal holdout" if final else "pooled development OOF; threshold metrics descriptive"
        rows.append(table)
    result = pd.concat(rows, ignore_index=True)
    folder = Path(next(iter(runs.values()))["bundle"]["frozen"]["config"]["suite_output_dir"])
    prefix = "FINAL" if final else "development"
    result.to_csv(folder / f"{prefix}_analysis_comparison.csv", index=False)
    models = result[result.model.isin(MODEL_NAMES)]
    plot_forest([f"{a} (n={n}, events={e})" for a, n, e in zip(models.analysis, models.n, models.events)],
        models.auroc.tolist(), models.auroc_lower_95.tolist(), models.auroc_upper_95.tolist(),
        folder / f"{prefix}_auroc_forest.png",
        "Temporal holdout AUROC by analysis" if final else "Development OOF AUROC by analysis",
        series=models.model.tolist())
    return result


def verify_ablation_pair(runs):
    """The ablation must use the same patients, split and folds as the main analysis."""
    if "filtered_last_hour" not in runs:
        return
    main, ablation = runs["filtered_six_hour"], runs["filtered_last_hour"]
    for field in ["development_ids", "test_ids"]:
        if main["bundle"][field] != ablation["bundle"][field]:
            raise ValueError("The feature ablation must use exactly the same patients and temporal split.")
    pd.testing.assert_frame_equal(main["oof"][["subject_id", "y_true", "outer_fold"]],
        ablation["oof"][["subject_id", "y_true", "outer_fold"]])


def paired_ablation_comparison(runs, final=False):
    """Six-hour mean minus last available hourly summary on identical patients."""
    main, ablation = runs["filtered_six_hour"], runs["filtered_last_hour"]
    file = "FINAL_test_predictions.csv" if final else "development_oof_predictions.csv"
    a = pd.read_csv(main["output_dir"] / file, dtype={"subject_id": "string"}).set_index("subject_id").sort_index()
    b = pd.read_csv(ablation["output_dir"] / file, dtype={"subject_id": "string"}).set_index("subject_id").sort_index()
    pd.testing.assert_series_equal(a.y_true, b.y_true)
    cfg = main["bundle"]["frozen"]["config"]
    rows = []
    for name in MODEL_NAMES:
        for r in paired_bootstrap(a.y_true, a[name], b[name], cfg["bootstrap_replicates"], cfg["seed"] + 400):
            rows.append({"model": name, "metric": r["metric"], "patients": len(a), "events": int(a.y_true.sum()),
                "six_hour": r["a"], "last_hour": r["b"], "six_hour_minus_last_hour": r["a_minus_b"],
                "lower_95": r["lower_95"], "upper_95": r["upper_95"], "valid_replicates": r["valid_replicates"]})
    table = pd.DataFrame(rows)
    table["interpretation"] = "paired patients; positive AUROC/AP or negative Brier differences favour six hours"
    prefix = "FINAL" if final else "development"
    table.to_csv(Path(cfg["suite_output_dir"]) / f"{prefix}_paired_feature_ablation.csv", index=False)
    return table


def sensitivity_row(r):
    """Name the two compared values in a paired_bootstrap result."""
    return {"metric": r["metric"], "a_value": r["a"], "b_value": r["b"], "a_minus_b": r["a_minus_b"],
        "lower_95": r["lower_95"], "upper_95": r["upper_95"], "valid_replicates": r["valid_replicates"]}


def missingness_filter_sensitivity(runs):
    """S2 and S3 on development patients: training-cohort effect vs change of patients."""
    if not {"filtered_six_hour", "full_six_hour"} <= set(runs):
        return None
    main, full = runs["filtered_six_hour"], runs["full_six_hour"]
    cfg = main["bundle"]["frozen"]["config"]
    reps, seed = cfg["bootstrap_replicates"], cfg["seed"] + 600
    main_oof = main["oof"].set_index("subject_id")
    full_oof = full["oof"].set_index("subject_id")
    rows = []
    # S2: identical main-cohort patients, scored out of fold by models trained on either cohort.
    shared = main_oof.index.intersection(full_oof.index)
    y_shared = main_oof.loc[shared, "y_true"]
    for name in MODEL_NAMES:
        for r in paired_bootstrap(y_shared, full_oof.loc[shared, name], main_oof.loc[shared, name], reps, seed):
            rows.append({"analysis": "S2_main_cohort_patients", "a": "full-cohort training (OOF)",
                "b": "filtered-cohort training (OOF)", "model": name, "patients": len(shared),
                "events": int(y_shared.sum()), **sensitivity_row(r)})
    # S3: development patients removed by the filter; the main-analysis models never saw them.
    removed = full["development"].loc[~full["development"].subject_id.isin(main_oof.index)]
    if len(removed) and removed[cfg["target"]].nunique() == 2:
        X = removed[main["bundle"]["frozen"]["selected_features"]]
        y_removed = removed[cfg["target"]].to_numpy()
        for name in MODEL_NAMES:
            filtered_scores = main["bundle"]["models"][name].predict_proba(X)[:, 1]
            full_scores = full_oof.loc[removed.subject_id, name].to_numpy()
            for r in paired_bootstrap(y_removed, full_scores, filtered_scores, reps, seed + 1):
                rows.append({"analysis": "S3_removed_patients", "a": "full-cohort training (OOF)",
                    "b": "filtered-cohort final model", "model": name, "patients": len(removed),
                    "events": int(y_removed.sum()), **sensitivity_row(r)})
    table = pd.DataFrame(rows)
    table.to_csv(Path(cfg["suite_output_dir"]) / "development_missingness_filter_sensitivity.csv", index=False)
    return table


In [ ]:
guard_final_suite(RUN_CONFIGS)
runs = {}
for analysis, cfg in RUN_CONFIGS.items():
    print(f"\n{analysis} ({cfg['run_mode']})", flush=True)
    runs[analysis] = run_development(cfg)
verify_ablation_pair(runs)

# Main comparison across analyses (pooled OOF with bootstrap intervals).
development_comparison = export_analysis_comparison(runs)
display(development_comparison[["analysis", "model", "n", "events", "mean_outer_fold_auroc", "auroc",
    "auroc_lower_95", "auroc_upper_95", "average_precision", "brier", "scaled_brier",
    "calibration_slope", "observed_expected_ratio", "sensitivity", "specificity", "ppv"]])
print("AUROC/AP are pooled OOF; mean_outer_fold_auroc selects the model. Threshold metrics are descriptive.")
display(Image(filename=str(SUITE_DIR / "development_auroc_forest.png")))

# Research question: six-hour trends vs last-hour snapshot on identical patients.
if "filtered_last_hour" in runs:
    display(paired_ablation_comparison(runs))

# S2 and S3: missingness-filter sensitivity on development patients.
filter_sensitivity = missingness_filter_sensitivity(runs)
if filter_sensitivity is not None:
    display(filter_sensitivity)

for analysis, run in runs.items():
    print(analysis, "selected model:", run["bundle"]["frozen"]["selected_primary_model"])
    print("S4: discrimination by outcome component")
    display(pd.read_csv(run["output_dir"] / "development_outcome_component_auroc.csv"))
    display(Image(filename=str(run["output_dir"] / "development_oof_curves.png")))
subgroup_figure = runs["filtered_six_hour"]["output_dir"] / "development_subgroup_auroc.png"
if subgroup_figure.exists():
    display(Image(filename=str(subgroup_figure)))

## 8. Interpretation

LR coefficients and XGBoost TreeSHAP importance are computed from the fitted base models on development patients. Numeric LR coefficients are log-odds per SD after fold-median imputation. SHAP contributions use base-model log odds before calibration; encoded categories are summed by source variable before taking mean absolute importance. Dependence plots show contribution against the supplied mean or hourly summary. These are predictive associations, not causal effects.


In [ ]:
def base_pipeline(model):
    """Uncalibrated tuned pipeline inside a calibrated model."""
    return model.calibrated_classifiers_[0].estimator if isinstance(model, CalibratedClassifierCV) else model


def transformed_variable(name, categorical):
    """Map a transformed column (numeric__RSBI_bin_3, categorical__gender_M) to its source variable."""
    prefix, _, column = name.partition("__")
    if prefix == "categorical":
        return next((c for c in categorical if column.startswith(c + "_")), column)
    return feature_family(column)


def export_interpretation(run):
    """LR coefficients, variable-level TreeSHAP importance and dependence plots."""
    bundle, dev, out = run["bundle"], run["development"], run["output_dir"]
    cfg = bundle["frozen"]["config"]
    features = bundle["frozen"]["selected_features"]
    categorical = [c for c in features if c in cfg["categorical_columns"]]
    numeric = [c for c in features if c not in categorical]

    lr = base_pipeline(bundle["models"]["logistic_regression"])
    names = lr.named_steps["preprocess"].get_feature_names_out()
    coefficients = pd.DataFrame({"feature": names, "variable": [transformed_variable(n, categorical) for n in names],
        "coefficient": lr.named_steps["model"].coef_[0]})
    coefficients["abs_coefficient"] = coefficients.coefficient.abs()
    coefficients = coefficients.sort_values("abs_coefficient", ascending=False)
    coefficients.to_csv(out / "logistic_base_coefficients.csv", index=False)

    # Native TreeSHAP contributions sum to the base model's log odds.
    xgb = base_pipeline(bundle["models"]["xgboost"])
    transformed = xgb.named_steps["preprocess"].transform(dev[features])
    booster = xgb.named_steps["model"].model_.get_booster()
    contrib = booster.predict(DMatrix(transformed), pred_contribs=True)
    margin = booster.predict(DMatrix(transformed), output_margin=True)
    np.testing.assert_allclose(contrib.sum(axis=1), margin, atol=1e-4, rtol=1e-4)
    columns = xgb.named_steps["preprocess"].get_feature_names_out()
    shap_frame = pd.DataFrame(contrib[:, :-1], columns=columns)
    pd.DataFrame({"feature": columns, "mean_abs_shap_log_odds": shap_frame.abs().mean().to_numpy()}).sort_values(
        "mean_abs_shap_log_odds", ascending=False).to_csv(out / "xgboost_base_treeshap_by_column.csv", index=False)
    # Combine encoded columns belonging to the same source variable.
    variables = np.array([transformed_variable(c, categorical) for c in columns])
    variable_shap = shap_frame.T.groupby(variables).sum().T
    importance = (variable_shap.abs().mean().sort_values(ascending=False)
        .rename("mean_abs_shap_log_odds").rename_axis("variable").reset_index())
    importance.to_csv(out / "xgboost_base_treeshap_by_variable.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 0.35 * len(importance) + 1.5))
    top = importance.iloc[::-1]
    ax.barh(top.variable, top.mean_abs_shap_log_odds)
    ax.set(xlabel="Mean absolute summed TreeSHAP contribution (base-model log odds)",
        title=f"{cfg['analysis']}: XGBoost variable importance, {len(dev)} development patients")
    fig.tight_layout()
    fig.savefig(out / "xgboost_base_treeshap.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    # Direction for the four most important numeric variables.
    pairs = []
    for v in importance.variable:
        column = next((c for c in numeric if feature_family(c) == v), None)
        if column:
            pairs.append((v, column))
    pairs = pairs[:4]
    if pairs:
        fig, axes = plt.subplots(1, len(pairs), figsize=(4 * len(pairs), 3.5), squeeze=False)
        for ax, (v, column) in zip(axes[0], pairs):
            x = dev[column].to_numpy(dtype=float)
            ax.scatter(x, variable_shap[v].to_numpy(), s=6, alpha=.4)
            ax.axhline(0, color="k", lw=.8)
            if np.isfinite(x).any():
                low, high = np.nanpercentile(x, [1, 99])
                if low < high:
                    ax.set_xlim(low, high)  # trim extreme values for readability
            ax.set(xlabel=f"{column} (supplied values)", ylabel=f"Summed SHAP: {v}", title=v)
        fig.tight_layout()
        fig.savefig(out / "xgboost_base_treeshap_dependence.png", dpi=180, bbox_inches="tight")
        plt.close(fig)
    return coefficients, importance


In [ ]:
for analysis, run in runs.items():
    print(analysis)
    lr_coefficients, xgb_importance = export_interpretation(run)
    display(lr_coefficients.head(10))
    display(Image(filename=str(run["output_dir"] / "xgboost_base_treeshap.png")))
    if (run["output_dir"] / "xgboost_base_treeshap_dependence.png").exists():
        display(Image(filename=str(run["output_dir"] / "xgboost_base_treeshap_dependence.png")))

## 9. Final evaluation (temporal holdout)

After trial and full development, enable `run_final_test` in main mode. Every analysis is frozen before any overlapping holdout is scored. Identical reruns reuse saved models/results; changed analyses are rejected after final evaluation starts.

All test patients are evaluated. Fewer than 20 events or non-events prints a warning; if one class is absent, metrics requiring both classes are recorded as undefined. Bootstrap files report the valid replicate count for each metric.

Outputs include metrics, ROC/PR/calibration/decision curves, patient-bootstrap intervals, paired model differences, outcome-component AUROCs and error/subgroup summaries. Intervals are conditional on fitted models. Filtered and full-cohort results are descriptive comparisons of different populations. Trial mode cannot open the holdout.


In [ ]:
def final_evaluation(run, enabled=False):
    """Score one frozen analysis on its temporal holdout."""
    if not enabled:
        print("Final test disabled. Enable it after fixing models, features and thresholds.")
        return None
    bundle, df, out = run["bundle"], run["data"], run["output_dir"]
    frozen = bundle["frozen"]
    cfg = frozen["config"]
    if cfg["run_mode"] != "main":
        raise ValueError("Final evaluation is disabled for trial runs. Complete a main run first.")
    marker = out / "final_test_started.json"
    signature = hashlib.sha256(json.dumps(frozen, sort_keys=True).encode()).hexdigest()
    model_hash = file_hash(out / "trained_models.joblib")
    if marker.exists():
        prior = json.loads(marker.read_text())
        if prior.get("analysis_signature") != signature or prior.get("model_sha256") != model_hash:
            raise RuntimeError("Models or settings changed after final evaluation started.")
        if prior["status"] == "completed":
            print("Loading saved final-test results.")
            return pd.read_csv(out / "FINAL_test_metrics.csv")
        print("Resuming final evaluation with the saved models.")
    test = df.set_index("subject_id", drop=False).loc[bundle["test_ids"]].reset_index(drop=True)
    warn_small_holdout(frozen["holdout"], cfg)
    # Save the marker before evaluation so interrupted runs can resume.
    marker_info = {"analysis_signature": signature, "model_sha256": model_hash,
        "selected_primary_model": frozen["selected_primary_model"], "holdout": frozen["holdout"]}
    write_json(marker, {**marker_info, "status": "started"})
    X = test[frozen["selected_features"]]
    scores = {n: m.predict_proba(X)[:, 1] for n, m in bundle["models"].items()}
    scores["prevalence_baseline"] = np.full(len(test), bundle["development_prevalence"])
    scores["random_guess"] = np.random.default_rng(cfg["seed"] + 300).uniform(size=len(test))
    p = prediction_table(test, cfg, scores, frozen["rsbi_column"])
    p.to_csv(out / "FINAL_test_predictions.csv", index=False)
    table = comparison_tables(p, frozen["thresholds"])
    table.to_csv(out / "FINAL_test_metrics.csv", index=False)
    plot_comparison(p, out, "FINAL_test", cfg["decision_curve_max_threshold"])
    if "rsbi" in p and p.loc[np.isfinite(p.rsbi), "y_true"].nunique() == 2:
        plot_comparison(p.loc[np.isfinite(p.rsbi)], out, "FINAL_test_rsbi_matched_subset",
            cfg["decision_curve_max_threshold"], include_rsbi=True)
    ci, differences = bootstrap_intervals(p, frozen["thresholds"], cfg)
    ci.to_csv(out / "FINAL_test_patient_bootstrap_CI.csv", index=False)
    differences.to_csv(out / "FINAL_paired_difference_XGB_minus_LR_CI.csv", index=False)
    outcome_component_auroc(p).to_csv(out / "FINAL_outcome_component_auroc.csv", index=False)

    # Error analysis with the frozen thresholds.
    for name in MODEL_NAMES:
        p[name + "_predicted_failure"] = (p[name] >= frozen["thresholds"][name]).astype(int)
        p[name + "_error"] = np.select([(p.y_true == 1) & (p[name + "_predicted_failure"] == 0),
            (p.y_true == 0) & (p[name + "_predicted_failure"] == 1)], ["false_negative", "false_positive"], default="correct")
    p.to_csv(out / "FINAL_test_error_analysis.csv", index=False)
    error_summary = []
    age_col = next((c for c in ["admission_age", "age"] if c in p), None)
    for name in MODEL_NAMES:
        for error, group in p.groupby(name + "_error"):
            row = {"model": name, "error": error, "patients": len(group),
                "death_events": int(group["death_within_48_hours"].sum()) if "death_within_48_hours" in group else np.nan,
                "reintubation_events": int(group["reintubation_within_48_hours"].sum()) if "reintubation_within_48_hours" in group else np.nan}
            if age_col:
                row["median_age"] = group[age_col].median()
            error_summary.append(row)
    pd.DataFrame(error_summary).to_csv(out / "FINAL_error_summary.csv", index=False)
    write_json(out / "FINAL_followup.json", followup_summary(test, cfg))
    subgroups = subgroup_metrics(p, frozen["thresholds"], cfg)
    subgroups.to_csv(out / "FINAL_exploratory_subgroup_metrics.csv", index=False)
    plot_subgroups(subgroups, out / "FINAL_subgroup_auroc.png", f"{cfg['analysis']}: holdout subgroups (exploratory)")
    write_json(marker, {**marker_info, "status": "completed",
        "CI_interpretation": "Conditional on fitted models and this test cohort; does not include uncertainty from repeating model development."})
    return table


def evaluate_suite(runs, configs, enabled=False):
    """Lock every analysis, then score every holdout regardless of event count."""
    if not enabled:
        print("Development complete for all analyses. Real holdouts remain unscored.")
        return None
    for cfg in configs.values():
        if cfg["run_mode"] != "main":
            raise ValueError("Final evaluation is unavailable in trial mode.")
    verify_ablation_pair(runs)
    guard_final_suite(configs)
    folder = Path(next(iter(configs.values()))["suite_output_dir"])
    marker = folder / "final_suite_started.json"
    if not marker.exists():
        for name, cfg in configs.items():
            frozen = runs[name]["bundle"]["frozen"]
            if cfg != frozen["config"] or load_cohort(cfg)[-1] != frozen["source_hashes"]:
                raise RuntimeError("Data or settings changed since development. Refit all analyses before opening the holdout.")
        write_json(marker, {"status": "started", "primary_analysis": "filtered_six_hour",
            "holdouts_evaluated": sorted(runs), "plan": suite_plan(configs),
            "model_hashes": {name: file_hash(run["output_dir"] / "trained_models.joblib") for name, run in runs.items()},
            "analysis_hashes": {name: file_hash(run["output_dir"] / "frozen_analysis.json") for name, run in runs.items()}})
    # The complete suite is locked BEFORE any overlapping holdout is scored.
    for name, run in runs.items():
        print("Final evaluation:", name)
        final_evaluation(run, enabled=True)
    result = export_analysis_comparison(runs, final=True)
    if {"filtered_six_hour", "filtered_last_hour"} <= set(runs):
        paired_ablation_comparison(runs, final=True)
    saved = json.loads(marker.read_text())
    write_json(marker, {**saved, "status": "completed"})
    return result


In [ ]:
final_results = evaluate_suite(
    runs, RUN_CONFIGS,
    enabled=CONFIG["execution"]["run_final_test"],
)
if final_results is not None:
    display(final_results)
    display(Image(filename=str(SUITE_DIR / "FINAL_auroc_forest.png")))
    for analysis, run in runs.items():
        if (run["output_dir"] / "FINAL_test_curves.png").exists():
            print(analysis)
            display(Image(filename=str(run["output_dir"] / "FINAL_test_curves.png")))
    if (SUITE_DIR / "FINAL_paired_feature_ablation.csv").exists():
        display(pd.read_csv(SUITE_DIR / "FINAL_paired_feature_ablation.csv"))


## Generative AI declaration

*To be completed by the group before submission; required in every deliverable. It must describe how AI was actually used.*

- Was a generative AI tool used? …
- Tool(s): …
- Purpose(s): …
- Statement on drafting text, generating code or creating content: …